In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║              FOMC MACRO DATASET — DATA LOADING PIPELINE                     ║
# ║  Produces a point-in-time safe panel indexed by FOMC meeting date.          ║
# ║  Every regressor reflects only information available on blackout_date.      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── CELL 1: SETUP ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
# ── CELL 2: INSTALL DEPENDENCIES ──────────────────────────────────────────────
!pip install fredapi pandas numpy requests openpyxl pyarrow joblib -q
!pip install nasdaq-data-link -q

In [3]:

# ── CELL 3: IMPORTS & GLOBAL CONFIG ───────────────────────────────────────────
import os
import re
import time
import random
import requests
import numpy  as np
import pandas as pd
from io       import BytesIO
from datetime import timedelta
from fredapi  import Fred
import nasdaqdatalink
import calendar


In [4]:

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = '/content/drive/My Drive/HEC Thesis'
STATEMENTS_DIR  = f'{BASE_DIR}/Text Data/fomc_statements'
SPEECHES_DIR    = f'{BASE_DIR}/Text Data/speeches'
SPEECH_SUBDIRS  = ['chair', 'vice_chair', 'others']
CALENDAR_PATH   = f'{BASE_DIR}/fomc_calendar.csv'
CACHE_DIR       = f'{BASE_DIR}/Data/cache'
OUT_DIR         = f'{BASE_DIR}/Data'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)

# ── FRED client ───────────────────────────────────────────────────────────────
FRED_API_KEY = 'f81255531fa6b64772aaab64d80e9b65'
fred         = Fred(api_key=FRED_API_KEY)

# Only pull vintages from this date forward (avoids pre-2010 irrelevant history
# and the pre-2000 FRED date-format quirk)
VINTAGE_START = '2010-01-01'

# Inflation variables that require a YoY transform in addition to levels
INFLATION_VARS = {'cpi', 'core_cpi', 'pce', 'core_pce'}



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 4: FOMC CALENDAR
# ══════════════════════════════════════════════════════════════════════════════

def _get_meeting_dates(statements_dir):
    """Extract FOMC meeting dates from statement filenames (format: statement_YYYYMMDD)."""
    dates = []
    for fname in os.listdir(statements_dir):
        m = re.search(r'statement_(\d{8})', fname)
        if m:
            dates.append(pd.to_datetime(m.group(1), format='%Y%m%d'))
    return sorted(dates)


def _get_speech_dates(speeches_dir, subdirs):
    """Return a DataFrame of all dated speech files across the given subdirectories."""
    records = []
    for subdir in subdirs:
        folder = os.path.join(speeches_dir, subdir)
        if not os.path.exists(folder):
            print(f"  Warning: folder not found → {folder}")
            continue
        for fname in os.listdir(folder):
            m = re.search(r'(\d{8})', fname)
            if m:
                records.append({
                    'date':   pd.to_datetime(m.group(1), format='%Y%m%d'),
                    'author': subdir,
                    'file':   fname,
                })
    return (pd.DataFrame(records)
              .drop_duplicates(subset=['date', 'file'])
              .sort_values('date')
              .reset_index(drop=True))


def _build_fomc_calendar(meeting_dates, speech_df, minutes_lag_days=21):
    """
    For each FOMC meeting derive three dates:
      meeting_date      — the FOMC decision day
      minutes_date      — meeting_date + 21 days (standard ~3-week release lag)
      blackout_date     — date of the LAST speech before this meeting
                          (proxy for when the Fed communication window closed)
      blackout_lag_days — days between blackout and meeting (sanity check)
    """
    all_speech_dates = sorted(speech_df['date'].unique())
    rows = []
    for meeting_dt in meeting_dates:
        minutes_dt      = meeting_dt + timedelta(days=minutes_lag_days)
        prior_speeches  = [d for d in all_speech_dates if d < meeting_dt]
        if prior_speeches:
            blackout_dt  = max(prior_speeches)
            blackout_lag = (meeting_dt - blackout_dt).days
        else:
            blackout_dt  = None
            blackout_lag = None
        rows.append({
            'meeting_date':      meeting_dt,
            'minutes_date':      minutes_dt,
            'blackout_date':     blackout_dt,
            'blackout_lag_days': blackout_lag,
        })
    return (pd.DataFrame(rows)
              .sort_values('meeting_date')
              .reset_index(drop=True))


def load_fomc_calendar():
    """
    Load the FOMC calendar from CSV if it exists, otherwise build it from
    the statement / speech text files and save it for future runs.
    """
    if os.path.exists(CALENDAR_PATH):
        print(f"[calendar] Loading from {CALENDAR_PATH}")
        df = pd.read_csv(CALENDAR_PATH, parse_dates=['meeting_date', 'minutes_date', 'blackout_date'])
        return df

    print("[calendar] CSV not found — building from text files ...")
    meeting_dates = _get_meeting_dates(STATEMENTS_DIR)
    speech_df     = _get_speech_dates(SPEECHES_DIR, SPEECH_SUBDIRS)
    print(f"  {len(meeting_dates)} meetings  |  {len(speech_df)} speeches")

    cal = _build_fomc_calendar(meeting_dates, speech_df)

    cal.to_csv(CALENDAR_PATH, index=False)
    print(f"  Saved to {CALENDAR_PATH}")

    # Sanity check: flag meetings where blackout was > 20 days ago
    suspicious = cal[cal['blackout_lag_days'] > 20]
    if not suspicious.empty:
        print(f"\n  ⚠  {len(suspicious)} meetings with blackout lag > 20 days "
              f"(possibly no speech in that inter-meeting window):")
        print(suspicious[['meeting_date', 'blackout_date', 'blackout_lag_days']].to_string(index=False))

    return cal


fomc_calendar = load_fomc_calendar()
print(fomc_calendar.to_string(index=False))



[calendar] Loading from /content/drive/My Drive/HEC Thesis/fomc_calendar.csv
 Unnamed: 0 meeting_date minutes_date blackout_date  blackout_lag_days
          0   2011-01-26   2011-02-16    2011-01-08                 18
          1   2011-03-15   2011-04-05    2011-03-04                 11
          2   2011-04-27   2011-05-18    2011-04-14                 13
          3   2011-06-22   2011-07-13    2011-06-14                  8
          4   2011-08-09   2011-08-30    2011-06-29                 41
          5   2011-09-21   2011-10-12    2011-09-15                  6
          6   2011-11-02   2011-11-23    2011-10-22                 11
          7   2011-12-13   2012-01-03    2011-11-29                 14
          8   2012-01-25   2012-02-15    2012-01-16                  9
          9   2012-03-13   2012-04-03    2012-03-01                 12
         10   2012-04-25   2012-05-16    2012-04-13                 12
         11   2012-06-20   2012-07-11    2012-06-12                  8


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 5: FRED HELPERS — VINTAGE MATRICES
# ══════════════════════════════════════════════════════════════════════════════

def _cache_path(name):
    return os.path.join(CACHE_DIR, f"{name}.parquet")


def _fred_get_with_retry(series_id, realtime_start, realtime_end,
                         max_retries=8, base_delay=2.0):
    """FRED get_series with exponential back-off for 429 / 403 / XML errors."""
    for attempt in range(max_retries):
        try:
            return fred.get_series(series_id,
                                   realtime_start=realtime_start,
                                   realtime_end=realtime_end)
        except Exception as e:
            err = str(e).lower()
            if 'mismatched tag' in err or 'parseerror' in err:
                return None
            is_rate  = 'too many requests' in err or '429' in err
            is_block = 'forbidden' in err or '403' in err
            if is_rate or is_block:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                if is_block and attempt < 2:
                    delay = max(delay, 30)
                print(f"\n    [retry {attempt+1}/{max_retries}] "
                      f"{'Rate-limited' if is_rate else 'Forbidden'} "
                      f"— waiting {delay:.0f}s ...", flush=True)
                time.sleep(delay)
            else:
                print(f"\n    Warning: unexpected error for {realtime_start}: {e}")
                return None
    print(f"\n    ERROR: gave up after {max_retries} retries for {realtime_start}")
    return None


def _fred_get_vintage_dates_with_retry(series_id, max_retries=8, base_delay=2.0):
    """FRED get_series_vintage_dates with the same back-off logic."""
    for attempt in range(max_retries):
        try:
            return fred.get_series_vintage_dates(series_id)
        except Exception as e:
            err = str(e).lower()
            is_rate  = 'too many requests' in err or '429' in err
            is_block = 'forbidden' in err or '403' in err
            is_xml   = 'mismatched tag' in err or 'parseerror' in err
            if is_rate or is_block or is_xml:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                if is_block and attempt < 2:
                    delay = max(delay, 30)
                print(f"\n    [retry metadata {attempt+1}/{max_retries}] "
                      f"Waiting {delay:.0f}s ...", flush=True)
                time.sleep(delay)
            else:
                print(f"\n    Warning: unexpected metadata error for {series_id}: {e}")
                return []
    print(f"\n    ERROR: gave up fetching vintage dates for {series_id}")
    return []


def build_vintage_matrix(series_id, name):
    """
    Pull the minimal set of FRED vintages needed to reconstruct the value
    known on every FOMC blackout date.  Results are cached to parquet.

    Returns a (obs_date × vintage_date) DataFrame of index levels.
    """
    final_path   = _cache_path(f"vintage_{name}")
    partial_path = _cache_path(f"vintage_{name}_partial")

    if os.path.exists(final_path):
        print(f"  [cache hit]  {name}")
        return pd.read_parquet(final_path)

    # All FRED vintage dates for this series, filtered to our sample window
    raw_dates = _fred_get_vintage_dates_with_retry(series_id)
    if not raw_dates:
        print(f"  ERROR: could not fetch vintage dates for {name}")
        return pd.DataFrame()

    all_vts = pd.to_datetime([
        d for d in raw_dates if pd.Timestamp(d) >= pd.Timestamp(VINTAGE_START)
    ])

    # For each blackout date, we need the vintage immediately on-or-before it
    # plus the next one as a safety buffer (forward-fill handles the gap)
    blackout_dates  = pd.to_datetime(fomc_calendar['blackout_date'].dropna().unique())
    needed_vintages = set()
    for bd in blackout_dates:
        before = all_vts[all_vts <= bd]
        after  = all_vts[all_vts  > bd]
        if not before.empty:
            needed_vintages.add(before[-1].strftime('%Y-%m-%d'))
        if not after.empty:
            needed_vintages.add(after[0].strftime('%Y-%m-%d'))

    vintage_dates = sorted(needed_vintages)
    print(f"  [fetching]   {name} — "
          f"{len(vintage_dates)} targeted vintages "
          f"(of {len(all_vts)} since {VINTAGE_START})", flush=True)

    # Resume from partial cache if available
    frames = {}
    if os.path.exists(partial_path):
        partial = pd.read_parquet(partial_path)
        frames  = {col.strftime('%Y-%m-%d'): partial[col] for col in partial.columns}
        remaining = [v for v in vintage_dates if v not in frames]
        print(f"    Resuming: {len(frames)} done, {len(remaining)} left", flush=True)
    else:
        remaining = vintage_dates

    for i, vdate in enumerate(remaining):
        result = _fred_get_with_retry(series_id, vdate, vdate)
        if result is not None:
            result.name    = pd.Timestamp(vdate)
            frames[vdate]  = result

        if (i + 1) % 20 == 0:
            print(f"    ... {i+1}/{len(remaining)}", flush=True)
            if frames:
                tmp         = pd.concat(frames.values(), axis=1)
                tmp.columns = pd.to_datetime(list(frames.keys()))
                tmp.to_parquet(partial_path)

        time.sleep(0.6)  # ~100 req/min — well under FRED's 120/min cap

    if not frames:
        print(f"  ERROR: no data retrieved for {name}")
        return pd.DataFrame()

    matrix         = pd.concat(frames.values(), axis=1)
    matrix.columns = pd.to_datetime(list(frames.keys()))
    matrix         = matrix.sort_index(axis=1)
    matrix.to_parquet(final_path)

    if os.path.exists(partial_path):
        os.remove(partial_path)

    print(f"  ✓  {name}  ({matrix.shape[1]} vintages × {matrix.shape[0]} obs)")
    return matrix


def compute_yoy_vintage_matrix(level_matrix, name):
    """
    Derive a YoY (%) vintage matrix from a level matrix.

    For each vintage column (= the series as known on that release date),
    compute the 12-month percent change.  This is fully vintage-consistent —
    it uses only what was published in that vintage, so there is zero look-ahead.

    The result is cached alongside the level matrix.
    """
    yoy_path = _cache_path(f"vintage_{name}_yoy")
    if os.path.exists(yoy_path):
        return pd.read_parquet(yoy_path)
    if level_matrix.empty:
        return pd.DataFrame()

    yoy_matrix = level_matrix.pct_change(periods=12) * 100
    yoy_matrix.to_parquet(yoy_path)
    return yoy_matrix


def vintage_as_of(matrix, blackout_date):
    """
    Extract (value, observation_date) from a vintage matrix as of blackout_date.
    Uses the latest vintage available on or before that date, then takes the
    most recent non-NaN observation in that snapshot.  Zero look-ahead.
    """
    if matrix.empty:
        return None, None
    bd             = pd.Timestamp(blackout_date)
    available_cols = matrix.columns[matrix.columns <= bd]
    if available_cols.empty:
        return None, None
    snapshot = matrix[available_cols].ffill(axis=1).iloc[:, -1]
    snapshot = snapshot[snapshot.index <= bd].dropna()
    if snapshot.empty:
        return None, None
    return snapshot.iloc[-1], snapshot.index[-1]



In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 6: SECONDARY DATA FETCHERS
# ══════════════════════════════════════════════════════════════════════════════
import os
ads_cache = '/content/drive/My Drive/HEC Thesis/Data/cache/ads_index.parquet'
os.remove(ads_cache)
print("Deleted ADS cache — re-run Cell 6 and Cell 9.")
# ── Market / Financial series (no revision risk) ──────────────────────────────

def fetch_market_series(series_id, name):
    """Download a non-revised FRED series once and cache it."""
    path = _cache_path(f"market_{name}")
    if os.path.exists(path):
        return pd.read_parquet(path).squeeze()
    print(f"  [fetching market]  {name}")
    s       = fred.get_series(series_id)
    s.index = pd.to_datetime(s.index)
    s.name  = name
    s.to_frame().to_parquet(path)
    return s


def market_as_of(series, blackout_date):
    """Latest non-NaN observation on or before blackout_date."""
    series.index = pd.to_datetime(series.index)
    sliced = series[series.index <= pd.Timestamp(blackout_date)].dropna()
    if sliced.empty:
        return None, None
    return sliced.iloc[-1], sliced.index[-1]


# ── ADS Business Conditions Index (Philadelphia Fed) ─────────────────────────

def fetch_ads():
    path = _cache_path("ads_index")
    if os.path.exists(path):
        print("  [cache hit]  ads_index")
        return pd.read_parquet(path).squeeze()

    print("  [fetching]   ADS Business Conditions Index ...")
    headers = {'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                              'AppleWebKit/537.36 (KHTML, like Gecko) '
                              'Chrome/120.0.0.0 Safari/537.36')}
    urls = [
        "https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/ads/ads_index_most_current_vintage.xlsx",
        "https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/ads/ads.xlsx",
    ]
    for url in urls:
        r = requests.get(url, headers=headers, timeout=30)
        if r.status_code == 200 and r.content.startswith(b'PK'):
            try:
                df = pd.read_excel(BytesIO(r.content), index_col=0, parse_dates=True, engine='openpyxl')
                df = df.iloc[:, [0]].rename(columns={df.columns[0]: 'ads'})
                df.to_parquet(path)
                return df['ads']
            except Exception as e:
                print(f"  Warning: could not parse ADS file: {e}")
    raise ValueError("Failed to fetch ADS index — Philadelphia Fed may be blocking the request.")


def ads_as_of(ads_series, blackout_date):
    ads_series.index = pd.to_datetime(ads_series.index)  # ← add this line
    sliced = ads_series[ads_series.index <= pd.Timestamp(blackout_date)].dropna()
    if sliced.empty:
        return None, None
    return sliced.iloc[-1], sliced.index[-1]


# ── Excess Bond Premium (Gilchrist & Zakrajšek / Fed) ────────────────────────

def fetch_ebp():
    path = _cache_path("excess_bond_premium")
    if os.path.exists(path):
        print("  [cache hit]  excess_bond_premium")
        return pd.read_parquet(path).squeeze()

    print("  [fetching]   Excess Bond Premium ...")
    url     = "https://www.federalreserve.gov/econresdata/notes/feds-notes/2016/files/ebp_csv.csv"
    headers = {'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                              'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36')}
    r = requests.get(url, headers=headers, timeout=30)
    if r.status_code != 200:
        raise ValueError(f"Failed to fetch EBP (HTTP {r.status_code})")
    df = pd.read_csv(BytesIO(r.content), parse_dates=['date'], index_col='date')
    df.to_parquet(path)
    return df['ebp']

def ebp_as_of(ebp_series, blackout_date):
    # Handle case where cache loaded as a DataFrame instead of a Series
    if isinstance(ebp_series, pd.DataFrame):
        ebp_series = ebp_series['ebp']
    ebp_series.index = pd.to_datetime(ebp_series.index)
    sliced = ebp_series[ebp_series.index <= pd.Timestamp(blackout_date)].dropna()
    if sliced.empty:
        return None, None
    return sliced.iloc[-1], sliced.index[-1]


# ── Survey of Professional Forecasters (Philadelphia Fed) ────────────────────

SPF_URLS = {
    'spf_gdp':   ('https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/'
                  'survey-of-professional-forecasters/data-files/files/median_rgdp_level.xlsx'),
    'spf_unemp': ('https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/'
                  'survey-of-professional-forecasters/data-files/files/median_unemp_level.xlsx'),
    'spf_cpi':   ('https://www.philadelphiafed.org/-/media/frbp/assets/surveys-and-data/'
                  'survey-of-professional-forecasters/data-files/files/median_cpi_level.xlsx'),
}


def fetch_spf(name, url):
    path = _cache_path(f"spf_{name}")
    if os.path.exists(path):
        print(f"  [cache hit]  {name}")
        return pd.read_parquet(path)
    print(f"  [fetching SPF]  {name} ...")
    r  = requests.get(url, timeout=30)
    df = pd.read_excel(BytesIO(r.content), engine='openpyxl')
    df.columns = df.columns.str.strip().str.upper()
    df['survey_date'] = (
        pd.to_datetime(df['YEAR'].astype(int).astype(str) + 'Q' +
                       df['QUARTER'].astype(int).astype(str))
        + pd.offsets.QuarterEnd(0)
    )
    df = df.sort_values('survey_date').set_index('survey_date')
    df.to_parquet(path)
    return df


def spf_as_of(spf_df, blackout_date):
    available = spf_df[spf_df.index <= pd.Timestamp(blackout_date)]
    if available.empty:
        return None, None
    return available.iloc[-1], available.index[-1]



Deleted ADS cache — re-run Cell 6 and Cell 9.


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 7: SERIES DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════

# Tier 1 — revised macro data: MUST use vintage matrices (look-ahead risk)
VINTAGE_SERIES = {
    'cpi':               'CPIAUCSL',   # CPI All Urban Consumers
    'core_cpi':          'CPILFESL',   # CPI ex Food & Energy
    'pce':               'PCEPI',      # PCE Price Index
    'core_pce':          'PCEPILFE',   # Core PCE Price Index
    'unemployment_rate': 'UNRATE',     # Unemployment Rate
    'nonfarm_payroll':   'PAYEMS',     # Nonfarm Payrolls
    'gdp':               'GDP',        # Nominal GDP
    'gdp_deflator':      'GDPDEF',     # GDP Deflator
    'nat_unemp_rate':    'NROU',       # CBO Natural Rate (for unemployment gap)
}

# Tier 2 — market / financial data: simple as-of slice, no revision risk
MARKET_SERIES = {
    'fed_funds_rate':   'FEDFUNDS',    # Effective Federal Funds Rate
    'yield_3mo':        'DTB3',        # 3-Month Treasury Bill
    'yield_6mo':        'DTB6',        # 6-Month Treasury Bill
    'yield_2yr':        'DGS2',        # 2-Year Treasury
    'yield_5yr':        'DGS5',        # 5-Year Treasury
    'yield_10yr':       'DGS10',       # 10-Year Treasury
    'vix':              'VIXCLS',      # VIX Volatility Index
    'breakeven_10yr':   'T10YIE',      # 10-Year Breakeven Inflation
    'term_spread_10_2': 'T10Y2Y',      # 10Y–2Y Yield Spread
    'real_rate_5yr':    'DFII5',       # 5-Year TIPS Real Rate
    'fed_target_midpoint': 'FEDTARMDLR',   # FOMC announced target midpoint (policy outcome)
}


In [ ]:


# ══════════════════════════════════════════════════════════════════════════════
# CELL 8: FETCH ALL DATA
# ══════════════════════════════════════════════════════════════════════════════

print("═" * 70)
print("STEP 1/5  —  FRED vintage matrices (levels)")
print("═" * 70)
vintage_matrices = {name: build_vintage_matrix(sid, name)
                    for name, sid in VINTAGE_SERIES.items()}

print("\n" + "═" * 70)
print("STEP 2/5  —  Inflation YoY matrices (derived from levels, vintage-safe)")
print("═" * 70)
# NOTE: YoY is computed *within* each vintage column, so it reflects exactly
# what was known at the time of that release — no look-ahead whatsoever.
# This avoids the FRED API limitation where units='pc1' cannot be combined
# with realtime_start / realtime_end parameters.
yoy_matrices = {
    name: compute_yoy_vintage_matrix(vintage_matrices[name], name)
    for name in INFLATION_VARS
}
print(f"  ✓  YoY matrices ready for: {', '.join(yoy_matrices.keys())}")

print("\n" + "═" * 70)
print("STEP 3/5  —  FRED market series")
print("═" * 70)
market_cache = {name: fetch_market_series(sid, name)
                for name, sid in MARKET_SERIES.items()}

print("\n" + "═" * 70)
print("STEP 4/5  —  ADS index & Excess Bond Premium")
print("═" * 70)
ads = fetch_ads()
ebp = fetch_ebp()

print("\n" + "═" * 70)
print("STEP 5/5  —  Survey of Professional Forecasters")
print("═" * 70)
spf_frames = {name: fetch_spf(name, url) for name, url in SPF_URLS.items()}

print("\n✅  All data fetched.")



══════════════════════════════════════════════════════════════════════
STEP 1/5  —  FRED vintage matrices (levels)
══════════════════════════════════════════════════════════════════════
  [cache hit]  cpi
  [cache hit]  core_cpi
  [cache hit]  pce
  [cache hit]  core_pce
  [cache hit]  unemployment_rate
  [cache hit]  nonfarm_payroll
  [cache hit]  gdp
  [cache hit]  gdp_deflator
  [cache hit]  nat_unemp_rate

══════════════════════════════════════════════════════════════════════
STEP 2/5  —  Inflation YoY matrices (derived from levels, vintage-safe)
══════════════════════════════════════════════════════════════════════
  ✓  YoY matrices ready for: cpi, core_cpi, pce, core_pce

══════════════════════════════════════════════════════════════════════
STEP 3/5  —  FRED market series
══════════════════════════════════════════════════════════════════════
  [fetching market]  fff_front_contract


ValueError: Bad Request.  The series does not exist.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9: ASSEMBLE STACKED DATASET
# ══════════════════════════════════════════════════════════════════════════════

print("═" * 70)
print("ASSEMBLING STACKED DATASET")
print("═" * 70)

# ── 1. Stack all three date types into a single timeline ──────────────────────
events = (
    pd.melt(
        fomc_calendar,
        value_vars=['blackout_date', 'meeting_date', 'minutes_date'],
        var_name='event_type',
        value_name='date'
    )
    .dropna(subset=['date'])
    .sort_values('date')
    .reset_index(drop=True)
)

# ── 2. For each event date, extract all macro data as-of that date ────────────
SPF_META_COLS = {'YEAR', 'QUARTER', 'SURVEY_DATE'}
rows = []

for _, ev in events.iterrows():
    date       = ev['date']
    event_type = ev['event_type']

    row = {'date': date, 'event_type': event_type}

    # Vintage macro levels + YoY for inflation vars
    for name, matrix in vintage_matrices.items():
        val, _ = vintage_as_of(matrix, date)
        row[name] = val
        if name in INFLATION_VARS:
            yoy_val, _ = vintage_as_of(yoy_matrices[name], date)
            row[f'{name}_yoy'] = yoy_val

    # Market / financial
    for name, series in market_cache.items():
        val, _ = market_as_of(series, date)
        row[name] = val

    # Excess bond premium
    row['excess_bond_prem'], _ = ebp_as_of(ebp, date)

    # SPF 1-quarter-ahead forecast
    for spf_name, spf_df in spf_frames.items():
        spf_row, _ = spf_as_of(spf_df, date)
        if spf_row is not None:
            fcol = next((c for c in spf_df.columns if c.upper() not in SPF_META_COLS), None)
            row[f'{spf_name}_1q'] = spf_row[fcol] if fcol else None
        else:
            row[f'{spf_name}_1q'] = None

    # Derived variables
    unemp        = row.get('unemployment_rate')
    nat_u        = row.get('nat_unemp_rate')
    core_pce_yoy = row.get('core_pce_yoy')
    y10          = row.get('yield_10yr')
    y2           = row.get('yield_2yr')

    row['unemployment_gap']          = (unemp - nat_u)       if pd.notna(unemp) and pd.notna(nat_u)   else None
    row['inflation_dev_from_target'] = (core_pce_yoy - 2.0)  if pd.notna(core_pce_yoy)                else None
    row['yield_spread_10_2']         = (y10 - y2)            if pd.notna(y10) and pd.notna(y2)        else None

    rows.append(row)

dataset = pd.DataFrame(rows).set_index('date')
dataset.index = pd.to_datetime(dataset.index)

print(f"Dataset shape: {dataset.shape}")
print(f"\nFirst 10 rows:")
display(dataset.head(10))

══════════════════════════════════════════════════════════════════════
ASSEMBLING STACKED DATASET
══════════════════════════════════════════════════════════════════════
Dataset shape: (363, 32)

First 10 rows:


,event_type,cpi,cpi_yoy,core_cpi,core_cpi_yoy,pce,pce_yoy,core_pce,core_pce_yoy,unemployment_rate,...,term_spread_10_2,real_rate_5yr,fff_implied_rate,excess_bond_prem,spf_gdp_1q,spf_unemp_1q,spf_cpi_1q,unemployment_gap,inflation_dev_from_target,yield_spread_10_2
date,,,,,,,,,,,,,,,,,,,,,
2011-01-08,blackout_date,219.146,0.000000,221.982,0.689005,111.493,0.611831,110.424,0.592126,9.4,...,2.74,0.05,NaN,-0.000186,13260.7,9.6,1.5,NaN,-1.407874,2.74
2011-01-26,meeting_date,220.252,0.000000,222.187,0.781991,111.493,0.611831,110.424,0.592126,9.4,...,2.83,0.07,NaN,-0.000186,13260.7,9.6,1.5,NaN,-1.407874,2.83
2011-02-16,minutes_date,220.186,1.206093,222.187,0.728991,111.869,0.911977,110.493,0.579850,9.0,...,2.76,0.43,NaN,-0.252483,13260.7,9.6,1.5,3.8,-1.420150,2.76
2011-03-04,blackout_date,221.062,1.183304,222.587,0.857292,112.166,1.021327,110.662,0.574389,8.9,...,2.81,-0.07,NaN,-0.294522,13260.7,9.6,1.5,3.7,-1.425611,2.81
2011-03-15,meeting_date,221.062,1.183304,222.587,0.857292,112.166,1.021327,110.662,0.574389,8.9,...,2.70,-0.16,NaN,-0.294522,13260.7,9.6,1.5,3.7,-1.425611,2.70
2011-04-05,minutes_date,222.270,1.176795,223.029,1.015916,112.601,1.442342,110.826,0.685921,8.8,...,2.66,-0.04,NaN,-0.242987,13382.6,9.6,2.6,3.6,-1.314079,2.66
2011-04-14,blackout_date,222.270,1.176795,223.029,1.015916,112.601,1.442342,110.826,0.685921,8.8,...,2.74,-0.02,NaN,-0.242987,13382.6,9.6,2.6,3.6,-1.314079,2.74
2011-04-27,meeting_date,223.490,1.176795,223.331,1.152700,112.601,1.442342,110.826,0.685921,8.8,...,2.74,-0.31,NaN,-0.242987,13382.6,9.6,2.6,3.6,-1.314079,2.74
2011-05-18,minutes_date,224.433,1.318793,223.745,1.223302,113.081,1.964798,110.996,0.733292,9.0,...,2.60,-0.27,NaN,-0.115163,13382.6,9.6,2.6,3.8,-1.266708,2.60


In [8]:
import pandas as pd
import numpy as np
import calendar

FOMC_CALENDAR_PATH = f'{BASE_DIR}/fomc_calendar.csv'
BLOOMBERG_PATH     = f'{BASE_DIR}/Data/FedFundsFutures.xlsx'
MASTER_PATH        = f'{BASE_DIR}/Data/Master_Macro.csv'

# =============================================================================
# 1. LOAD FOMC CALENDAR
# =============================================================================
fomc = pd.read_csv(FOMC_CALENDAR_PATH, parse_dates=["meeting_date", "minutes_date", "blackout_date"])

next_meeting_placeholder = pd.DataFrame([{
    "meeting_date":  pd.Timestamp("2026-01-28"),
    "minutes_date":  pd.NaT,
    "blackout_date": pd.NaT
}])

fomc = pd.concat([fomc, next_meeting_placeholder], ignore_index=True)
fomc = fomc.sort_values("meeting_date").reset_index(drop=True)
fomc["next_meeting_date"] = fomc["meeting_date"].shift(-1)

records = []
for _, row in fomc.iterrows():
    if pd.isna(row["next_meeting_date"]):
        continue

    # POST-meeting dates → price the NEXT meeting
    for col in ["meeting_date", "minutes_date"]:
        date = row[col]
        if pd.notna(date):
            records.append({
                "rep_date":          date,
                "next_meeting_date": row["next_meeting_date"],
            })

    # PRE-meeting date → price THIS row's meeting
    if pd.notna(row["blackout_date"]):
        records.append({
            "rep_date":          row["blackout_date"],
            "next_meeting_date": row["meeting_date"],  # same row, not next
        })

rep_df = (pd.DataFrame(records)
            .drop_duplicates("rep_date")
            .sort_values("rep_date")
            .reset_index(drop=True))

# =============================================================================
# 2. LOAD BLOOMBERG DATA
# =============================================================================
bb = pd.read_excel(BLOOMBERG_PATH, header=0)
bb.columns = ["date_ffr", "target_ffr", "date_effr", "effr", "date_ff1", "FF1", "date_ff2", "FF2", "date_ff3", "FF3"]

for col in ["date_ffr", "date_effr", "date_ff1", "date_ff2", "date_ff3"]:
    bb[col] = pd.to_datetime(bb[col], errors="coerce")

for col in ["target_ffr", "effr", "FF1", "FF2", "FF3"]:
    bb[col] = pd.to_numeric(bb[col], errors="coerce")

for col in ["FF1", "FF2", "FF3"]:
    bb[col] = 100 - bb[col]

ffr_df  = bb[["date_ffr",  "target_ffr"]].dropna().sort_values("date_ffr")
effr_df = bb[["date_effr", "effr"]].dropna().sort_values("date_effr")
ff1_df  = bb[["date_ff1",  "FF1"]].dropna().sort_values("date_ff1")
ff2_df  = bb[["date_ff2",  "FF2"]].dropna().sort_values("date_ff2")
ff3_df  = bb[["date_ff3",  "FF3"]].dropna().sort_values("date_ff3")

# =============================================================================
# 3. MERGE EACH SERIES ONTO REPRESENTATIVE DATES
# =============================================================================
rep_df = rep_df.sort_values("rep_date")
rep_df = pd.merge_asof(rep_df, ffr_df,  left_on="rep_date", right_on="date_ffr",  direction="backward").drop(columns="date_ffr")
rep_df = pd.merge_asof(rep_df, effr_df, left_on="rep_date", right_on="date_effr", direction="backward").drop(columns="date_effr")
rep_df = pd.merge_asof(rep_df, ff1_df,  left_on="rep_date", right_on="date_ff1",  direction="backward").drop(columns="date_ff1")
rep_df = pd.merge_asof(rep_df, ff2_df,  left_on="rep_date", right_on="date_ff2",  direction="backward").drop(columns="date_ff2")
rep_df = pd.merge_asof(rep_df, ff3_df,  left_on="rep_date", right_on="date_ff3",  direction="backward").drop(columns="date_ff3")

# =============================================================================
# 4. IDENTIFY MONTHS AHEAD
# =============================================================================
def months_between(d1, d2):
    return (d2.year - d1.year) * 12 + (d2.month - d1.month)

rep_df["months_ahead"] = rep_df.apply(
    lambda r: months_between(r["rep_date"], r["next_meeting_date"]), axis=1)

# =============================================================================
# 5. ADVANCED KUTTNER CORRECTION
# =============================================================================
def kuttner_advanced(row):
    mtg = row["next_meeting_date"]
    if pd.isna(mtg): return np.nan

    m = row["months_ahead"]

    if m == 0:
        f, r0, f_next = row["FF1"], row["effr"], row["FF2"]
    elif m == 1:
        f, r0, f_next = row["FF2"], row["FF1"],  row["FF3"]
    elif m == 2:
        f, r0, f_next = row["FF3"], row["FF2"],  np.nan
    else:
        return np.nan

    if pd.isna(f) or pd.isna(r0):
        return np.nan

    D  = calendar.monthrange(mtg.year, mtg.month)[1]
    d1 = mtg.day - 1
    d2 = D - d1

    if d2 <= 0:
        return np.nan

    if d2 <= 7:
        return round(f_next, 4) if pd.notna(f_next) else round(f, 4)

    return round((D * f - d1 * r0) / d2, 4)

rep_df["implied_ffr"] = rep_df.apply(kuttner_advanced, axis=1)

# =============================================================================
# 6. OUTPUT
# =============================================================================
output = rep_df[["rep_date", "next_meeting_date", "implied_ffr",
                 "months_ahead", "target_ffr", "effr", "FF1"]].copy()

print(output.head(20).to_string(index=False))
print(f"\nShape: {output.shape}")
print(f"NaN implied_ffr: {output['implied_ffr'].isna().sum()} rows")

output.to_csv(f"{BASE_DIR}/kuttner_implied_ffr.csv", index=False)
print(f"Saved to {BASE_DIR}/kuttner_implied_ffr.csv")

# =============================================================================
# 7. APPEND TO MASTER MACRO
# =============================================================================
try:
    master = pd.read_csv(MASTER_PATH, parse_dates=["date"])
    rep_df["rep_date"] = pd.to_datetime(rep_df["rep_date"])

    implied = rep_df[["rep_date", "implied_ffr"]].rename(columns={"rep_date": "date"})

    # Index check: all implied dates must exist in master before merging
    master_dates  = set(master["date"])
    implied_dates = set(implied["date"])
    missing = implied_dates - master_dates

    if missing:
        print(
            f"ERROR: {len(missing)} rep_date(s) from kuttner output are not present "
            f"in Master_Macro. Merge aborted. Missing dates:\n{sorted(missing)}"
        )
    else:
        if "implied_ffr" in master.columns:
            master = master.drop(columns=["implied_ffr"])

        master = master.merge(implied, on="date", how="left")
        master.to_csv(MASTER_PATH, index=False)
        print(f"Success. Clean implied_ffr appended and saved to {MASTER_PATH}")

except FileNotFoundError:
    print(f"Master file not found at {MASTER_PATH}. Skipping append step.")

  rep_date next_meeting_date  implied_ffr  months_ahead  target_ffr  effr    FF1
2011-01-08        2011-01-26       0.1800             0        0.25  0.17 0.1775
2011-01-26        2011-03-15       0.1609             2        0.25  0.17 0.1700
2011-02-16        2011-03-15       0.1488             1        0.25  0.15 0.1625
2011-03-04        2011-03-15       0.1500             0        0.25  0.15 0.1500
2011-03-15        2011-04-27       0.1400             1        0.25  0.14 0.1425
2011-04-05        2011-04-27       0.1200             0        0.25  0.09 0.1100
2011-04-14        2011-04-27       0.1050             0        0.25  0.09 0.1000
2011-04-27        2011-06-22       0.1433             2        0.25  0.09 0.1000
2011-05-18        2011-06-22       0.1175             1        0.25  0.10 0.0925
2011-06-14        2011-06-22       0.1000             0        0.25  0.10 0.1000
2011-06-22        2011-08-09       0.1320             2        0.25  0.09 0.0975
2011-06-29        2011-08-09

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 10: SAVE
# ══════════════════════════════════════════════════════════════════════════════

dataset.to_parquet(os.path.join(OUT_DIR, 'fomc_macro_dataset.parquet'))
dataset.to_csv(os.path.join(OUT_DIR, 'fomc_macro_dataset.csv'))

print(f"\n✅  Saved to {OUT_DIR}")
print(f"   fomc_macro_dataset.parquet")
print(f"   fomc_macro_dataset.csv")


✅  Saved to /content/drive/My Drive/HEC Thesis/Data
   fomc_macro_dataset.parquet
   fomc_macro_dataset.csv


In [7]:
import pandas as pd

MASTER_PATH = f'{BASE_DIR}/Data/Master_Macro.csv'

# Load Master Macro
master = pd.read_csv(MASTER_PATH, parse_dates=["date"])

# Ensure rep_df date column is also datetime (should already be, but just in case)
rep_df["rep_date"] = pd.to_datetime(rep_df["rep_date"])

# Check if dates match exactly
master_dates = set(master["date"])
rep_dates    = set(rep_df["rep_date"])

if master_dates != rep_dates:
    in_master_not_rep = master_dates - rep_dates
    in_rep_not_master = rep_dates - master_dates
    print("Dates do not match. Merge aborted.")
    if in_master_not_rep:
        print(f"\n{len(in_master_not_rep)} date(s) in Master_Macro but NOT in rep_df:")
        for d in sorted(in_master_not_rep):
            print(f"  {d.date()}")
    if in_rep_not_master:
        print(f"\n{len(in_rep_not_master)} date(s) in rep_df but NOT in Master_Macro:")
        for d in sorted(in_rep_not_master):
            print(f"  {d.date()}")
else:
    # Dates match exactly — merge and overwrite
    implied = rep_df[["rep_date", "implied_ffr"]].rename(columns={"rep_date": "date"})
    master  = master.merge(implied, on="date", how="left")
    master.to_csv(MASTER_PATH, index=False)
    print(f"Success. implied_ffr appended and saved to {MASTER_PATH}")

MergeError: Passing 'suffixes' which cause duplicate columns {'implied_ffr_x'} is not allowed.